# 02 – Préparation

Objectif : transformer les données brutes en données propres, prêtes pour les modèles. On corrige les problèmes repérés dans `01_exploration` (voir le tableau « Bilan » à la fin de ce notebook).

**Règle d'or : on sépare les données en train / validation / test *avant* tout traitement.**

### Pourquoi séparer d'abord ? La fuite de données

Le jeu de **test** sert à simuler de nouveaux clients que le modèle n'a jamais vus. Il doit donc rester « inconnu » jusqu'au bout.

Imaginons qu'on remplace les revenus manquants par la médiane calculée sur *tout* le fichier. Cette médiane contiendrait un peu d'information venant des clients du test. Le modèle aurait alors « vu » le test sans le savoir, et ses résultats sur le test seraient un peu trop optimistes. C'est ce qu'on appelle une **fuite de données** (*data leakage*).

La bonne méthode :
1. on découpe les données en trois jeux ;
2. on **calcule** toutes les valeurs utiles (médianes, plafonds…) **uniquement sur le train** ;
3. on **applique** ces mêmes valeurs, sans les recalculer, à la validation et au test.

### Les trois jeux

| Jeu | Part | À quoi il sert |
|---|---|---|
| **Train** (entraînement) | 60 % | Le modèle apprend sur ces clients. Toutes les médianes et plafonds sont calculés ici. |
| **Validation** | 20 % | Comparer des réglages et des modèles, choisir le seuil de décision. On peut le consulter souvent. |
| **Test** | 20 % | Mesurer la performance finale, **une seule fois**, à la toute fin. |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.preparation import (
    TARGET, LATE, COUNTS, CONTINUOUS, LATE_CODES, AGE_MIN, RU_ERROR, RU_CAP,
    INCOME_MIN, CAP_QUANTILE, split_stratified, Preparateur,
)

DATA = '../data/cs-training.csv'
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
pd.set_option('display.max_columns', 20)

Le code de préparation est rangé dans [`src/preparation.py`](../src/preparation.py) pour pouvoir être réutilisé tel quel dans le notebook de modélisation. Ce notebook explique et justifie chaque choix, puis l'applique.

## 1. Chargement et suppression des doublons

In [2]:
df = pd.read_csv(DATA, index_col=0)
n_avant = len(df)
df = df.drop_duplicates()
print(f"Lignes avant : {n_avant:,}  |  doublons supprimés : {n_avant - len(df):,}  |  lignes restantes : {len(df):,}")

Lignes avant : 150,000  |  doublons supprimés : 609  |  lignes restantes : 149,391


**Pourquoi supprimer les doublons *avant* la séparation, alors qu'on a dit « séparer avant tout traitement » ?**

La règle concerne les traitements qui **apprennent quelque chose des données** (une médiane, un plafond…). Supprimer une ligne identique à une autre n'apprend rien : on ne calcule aucune valeur.

Au contraire, garder les doublons créerait une fuite : une ligne pourrait se retrouver dans le train **et** sa copie dans le test. Le modèle serait alors testé sur un client qu'il a déjà vu.

## 2. Séparation stratifiée train / validation / test

**Stratifiée** veut dire que chaque jeu garde la même proportion de défauts (environ 6,7 %). Sans cela, un jeu pourrait par hasard contenir un peu plus ou un peu moins de mauvais clients, ce qui fausserait les comparaisons.

`random_state=42` fixe le hasard : on obtient exactement le même découpage à chaque exécution.

In [3]:
train_raw, val_raw, test_raw = split_stratified(df, val_size=0.2, test_size=0.2, random_state=42)

pd.DataFrame({
    'nb_clients': [len(train_raw), len(val_raw), len(test_raw)],
    'part': [len(train_raw) / len(df), len(val_raw) / len(df), len(test_raw) / len(df)],
    'nb_defauts': [train_raw[TARGET].sum(), val_raw[TARGET].sum(), test_raw[TARGET].sum()],
    'taux_defaut': [train_raw[TARGET].mean(), val_raw[TARGET].mean(), test_raw[TARGET].mean()],
}, index=['train', 'validation', 'test'])

,nb_clients,part,nb_defauts,taux_defaut
train,89634,0.600,6005,0.067
validation,29878,0.200,2002,0.067
test,29879,0.200,2002,0.067


Les trois jeux ont bien le même taux de défaut.

**À partir d'ici, toutes les analyses qui servent à prendre une décision sont faites sur `train_raw` uniquement.** On ne regarde plus la validation ni le test.

## 3. Traitement des valeurs bizarres et des manquants

Pour chaque problème du bilan de l'exploration, on présente le choix fait et sa justification. Les choix sont de deux types :
- des **règles fixes**, décidées à partir du sens des variables (par exemple « un âge inférieur à 18 ans est impossible »). Elles n'apprennent rien des données, donc pas de risque de fuite ;
- des **valeurs apprises sur le train** : médianes pour remplir les trous, 99ᵉ percentile pour plafonner.

Le principe général pour une valeur fausse : on la remplace par une **valeur manquante (NaN)**, puis tous les NaN sont remplis de la même façon, par la **médiane du train**. On utilise la médiane plutôt que la moyenne, car la moyenne est tirée vers le haut par les valeurs extrêmes (par exemple, la moyenne des revenus est gonflée par quelques revenus de plusieurs millions).

### 3.1 Codes 96 / 98 dans les colonnes de retards

In [4]:
code_train = train_raw[LATE].isin(LATE_CODES).any(axis=1)
print(f"Clients avec un code 96/98 dans le train : {code_train.sum()}")
print(f"Taux de défaut de ces clients : {train_raw.loc[code_train, TARGET].mean():.1%}  (moyenne : {train_raw[TARGET].mean():.1%})")

Clients avec un code 96/98 dans le train : 123
Taux de défaut de ces clients : 61.8%  (moyenne : 6.7%)


**Choix.**
1. On crée une colonne **`retards_code_special`** qui vaut 1 pour ces clients et 0 pour les autres.
2. Dans les trois colonnes de retards, on remplace 96 et 98 par NaN, puis par la médiane du train (0 retard).

**Justification.** 96 et 98 ne sont pas des nombres de retards. Les garder ferait croire au modèle que ces clients ont 30 fois plus de retards que les autres. Mais ces clients font défaut environ 60 % du temps : l'information est précieuse. La colonne indicatrice la conserve, sans fausser les colonnes de retards.

### 3.2 Âge impossible

In [5]:
print(f"Clients de moins de {AGE_MIN} ans dans le train : {(train_raw['age'] < AGE_MIN).sum()}")
print(f"Âge médian du train : {train_raw['age'].median():.0f} ans")

Clients de moins de 18 ans dans le train : 1
Âge médian du train : 52 ans


**Choix.** Un âge inférieur à 18 ans est remplacé par l'âge médian du train.

**Justification.** On ne peut pas emprunter avant 18 ans : c'est une erreur de saisie. Il y a au plus un client concerné, donc le choix n'a aucun effet mesurable. Remplacer plutôt que supprimer permet de traiter de la même façon un éventuel futur client avec la même erreur (on ne peut pas « supprimer » un client à qui il faut donner une réponse).

### 3.3 Utilisation du crédit renouvelable aberrante

In [6]:
ru = train_raw['RevolvingUtilizationOfUnsecuredLines']
tranches = pd.cut(ru, [-0.01, 1, RU_CAP, RU_ERROR, np.inf],
                  labels=['0-1', f'1-{RU_CAP}', f'{RU_CAP}-{RU_ERROR}', f'> {RU_ERROR}'])
train_raw.groupby(tranches, observed=True)[TARGET].agg(nb_clients='size', taux_defaut='mean')

,nb_clients,taux_defaut
RevolvingUtilizationOfUnsecuredLines,,
0-1,87639,0.060
1-2,1758,0.403
2-10,83,0.253
> 10,154,0.065


**Choix.**
1. Les valeurs **supérieures à 10** sont remplacées par NaN, puis par la médiane du train.
2. Les valeurs restantes sont **plafonnées à 2** : un client à 3 ou à 8 est traité comme un client à 2.

**Justification.**
- Au-delà de 10, le taux de défaut retombe au niveau de la moyenne : ces clients ne se comportent pas comme des clients qui dépassent leur plafond. La valeur est fausse, donc on la traite comme inconnue. La médiane est un remplacement raisonnable puisque ces clients ont un risque proche de la moyenne.
- Entre 1 et 2 (le client a dépassé son plafond), le risque est très élevé : environ 40 %. Entre 2 et 10, il reste élevé (environ 25 %), mais ce groupe ne compte qu'environ 80 clients dans le train : trop peu pour que le modèle apprenne un comportement fiable. On les regroupe donc avec les clients à 2. Le plafond de 2 (deux fois le plafond autorisé) garde la distinction importante : « au-dessus de 1 = dépassement ».

### 3.4 Revenus manquants, revenus de 0 ou 1

In [7]:
inc = train_raw['MonthlyIncome']
groupe = np.select([inc.isna(), inc <= INCOME_MIN], ['manquant', '0 ou 1'], 'normal')
train_raw.groupby(groupe)[TARGET].agg(nb_clients='size', taux_defaut='mean')

,nb_clients,taux_defaut
0 ou 1,1335,0.037
manquant,17525,0.055
normal,70774,0.071


**Choix.**
1. On crée deux colonnes indicatrices : **`revenu_manquant`** (revenu non renseigné) et **`revenu_0_1`** (revenu égal à 0 ou 1).
2. Les revenus de 0 ou 1 sont remplacés par NaN, puis tous les revenus manquants par la médiane du train.

**Justification.**
- Les clients sans revenu font **moins** défaut que les autres, et ceux à 0 ou 1 encore moins. Le fait que le revenu manque est donc une information : les indicateurs la gardent. Remplacer simplement par la médiane, sans indicateur, ferait disparaître cette information.
- Deux indicateurs plutôt qu'un seul, parce que les deux groupes n'ont pas le même risque (environ 3,7 % contre 5,7 %).
- Un revenu de 1 $ par mois n'est pas réaliste : c'est une valeur saisie par défaut, qu'on traite comme un revenu inconnu.

### 3.5 Taux d'endettement quand le revenu n'est pas fiable

Dans l'exploration, on a vu que sans revenu, `DebtRatio` contient un montant en dollars et non un rapport. Peut-on récupérer l'information en recalculant un vrai rapport (dépenses ÷ revenu médian) ?

In [8]:
sans_revenu = inc.isna() | (inc <= INCOME_MIN)
dr = train_raw['DebtRatio']
ratio_recalcule = dr[sans_revenu] / inc[~sans_revenu].median()

print("Clients SANS revenu fiable : taux de défaut par tranche de rapport recalculé")
print(train_raw[sans_revenu].groupby(pd.qcut(ratio_recalcule, 5), observed=True)[TARGET].mean().round(3), '\n')
print("Clients AVEC revenu : taux de défaut par tranche de taux d'endettement")
print(train_raw[~sans_revenu].groupby(pd.qcut(dr[~sans_revenu], 5), observed=True)[TARGET].mean().round(3))

Clients SANS revenu fiable : taux de défaut par tranche de rapport recalculé
DebtRatio
(-0.001, 0.0129]   0.046
(0.0129, 0.134]    0.077
(0.134, 0.291]     0.051
(0.291, 0.493]     0.042
(0.493, 59.939]    0.051
Name: SeriousDlqin2yrs, dtype: float64 

Clients AVEC revenu : taux de défaut par tranche de taux d'endettement
DebtRatio
(-0.001, 0.105]    0.060
(0.105, 0.233]     0.059
(0.233, 0.353]     0.054
(0.353, 0.523]     0.070
(0.523, 450.333]   0.108
Name: SeriousDlqin2yrs, dtype: float64


**Constat.** Chez les clients **avec** revenu, le risque augmente nettement avec l'endettement. Chez les clients **sans** revenu, même recalculé, le rapport ne montre pas de tendance claire : le taux de défaut monte et descend d'une tranche à l'autre sans logique. Le recalcul n'apporte donc pas d'information fiable.

**Choix.** Pour les clients sans revenu fiable (manquant, 0 ou 1), `DebtRatio` est remplacé par NaN, puis par la médiane du train.

**Justification.** La colonne mélange deux choses différentes (un rapport et un montant en $). La garder telle quelle ferait croire au modèle que ces clients sont « 1 000 fois plus endettés ». Les indicateurs `revenu_manquant` et `revenu_0_1` signalent déjà au modèle que ce sont des clients à part.

### 3.6 Personnes à charge manquantes

In [9]:
dep_na = train_raw['NumberOfDependents'].isna()
print(f"Personnes à charge manquantes : {dep_na.sum():,}")
print(f"... dont revenu aussi manquant : {(dep_na & train_raw['MonthlyIncome'].isna()).sum():,}")
print(f"Médiane du train : {train_raw['NumberOfDependents'].median():.0f}")

Personnes à charge manquantes : 2,303
... dont revenu aussi manquant : 2,303
Médiane du train : 0


**Choix.** Remplacement par la médiane du train (0 personne à charge), **sans** créer d'indicateur supplémentaire.

**Justification.** Tous ces clients ont aussi un revenu manquant : l'indicateur `revenu_manquant` porte déjà cette information. Un deuxième indicateur serait presque une copie du premier.

### 3.7 Valeurs extrêmes : plafonnement au 99ᵉ percentile

Pour les autres variables (revenu, taux d'endettement, comptages), on plafonne au **99ᵉ percentile du train** : les 1 % de valeurs les plus grandes sont ramenées à cette limite. C'est l'équivalent de la tranche « N et + » des graphiques de l'exploration.

**Justification.** Ces valeurs extrêmes (54 prêts immobiliers, 3 millions de revenu par mois…) sont rares et peu fiables. Sans plafond, elles pèseraient beaucoup trop dans la régression logistique. Le 99ᵉ percentile est calculé **sur le train seulement**, puis appliqué tel quel à la validation et au test. L'âge n'est pas plafonné : ses grandes valeurs (jusqu'à 109 ans) restent plausibles.

## 4. Application : apprendre sur le train, appliquer partout

`fit` calcule les médianes et les plafonds **sur le train uniquement**. `transform` applique les corrections avec ces mêmes valeurs aux trois jeux.

In [10]:
prep = Preparateur().fit(train_raw)
train = prep.transform(train_raw)
val = prep.transform(val_raw)
test = prep.transform(test_raw)

pd.DataFrame({'mediane (remplissage)': prep.medianes_, 'plafond': prep.plafonds_})

,mediane (remplissage),plafond
RevolvingUtilizationOfUnsecuredLines,0.153,2.000
age,52.000,inf
DebtRatio,0.290,2.026
MonthlyIncome,"5,500.000","25,000.000"
NumberOfTime30-59DaysPastDueNotWorse,0.000,3.000
NumberOfTime60-89DaysPastDueNotWorse,0.000,2.000
NumberOfTimes90DaysLate,0.000,2.000
NumberOfOpenCreditLinesAndLoans,8.000,24.000
NumberRealEstateLoansOrLines,1.000,5.000
NumberOfDependents,0.000,4.000


Ce tableau contient **tout ce qui a été appris des données**. Toutes ces valeurs viennent du train.

## 5. Vérifications

In [11]:
for nom, d in [('train', train), ('validation', val), ('test', test)]:
    print(f"{nom:<11} : {d.shape[0]:>6,} lignes, {d.shape[1]} colonnes, valeurs manquantes restantes = {d.isna().sum().sum()}")

train       : 89,634 lignes, 14 colonnes, valeurs manquantes restantes = 0
validation  : 29,878 lignes, 14 colonnes, valeurs manquantes restantes = 0
test        : 29,879 lignes, 14 colonnes, valeurs manquantes restantes = 0


In [12]:
# Avant / après sur le train : les valeurs aberrantes ont disparu
features = CONTINUOUS + COUNTS
pd.concat({
    'min avant': train_raw[features].min(), 'max avant': train_raw[features].max(),
    'min après': train[features].min(), 'max après': train[features].max(),
}, axis=1)

,min avant,max avant,min après,max après
RevolvingUtilizationOfUnsecuredLines,0.000,"50,708.000",0.000,2.000
age,0.000,109.000,21.000,109.000
DebtRatio,0.000,"329,664.000",0.000,2.026
MonthlyIncome,0.000,"3,008,750.000",2.000,"25,000.000"
NumberOfTime30-59DaysPastDueNotWorse,0.000,98.000,0.000,3.000
NumberOfTime60-89DaysPastDueNotWorse,0.000,98.000,0.000,2.000
NumberOfTimes90DaysLate,0.000,98.000,0.000,2.000
NumberOfOpenCreditLinesAndLoans,0.000,58.000,0.000,24.000
NumberRealEstateLoansOrLines,0.000,54.000,0.000,5.000
NumberOfDependents,0.000,13.000,0.000,4.000


In [13]:
# Les nouvelles colonnes indicatrices : combien de clients et quel taux de défaut ?
indicateurs = ['retards_code_special', 'revenu_manquant', 'revenu_0_1']
pd.DataFrame({
    col: {'nb_clients (train)': train[col].sum(),
          'taux_defaut si 1': train.loc[train[col] == 1, TARGET].mean(),
          'taux_defaut si 0': train.loc[train[col] == 0, TARGET].mean()}
    for col in indicateurs
}).T

,nb_clients (train),taux_defaut si 1,taux_defaut si 0
retards_code_special,123.000,0.618,0.066
revenu_manquant,"17,525.000",0.055,0.070
revenu_0_1,"1,335.000",0.037,0.067


**À retenir.**
- Il ne reste **aucune valeur manquante** dans les trois jeux.
- Les maximums aberrants ont disparu (âge minimum 21 ans, utilisation du crédit au plus 2, plus de codes 96/98).
- Les trois indicateurs ont un taux de défaut très différent selon qu'ils valent 0 ou 1 : ils apportent bien de l'information au modèle.

## 6. Sauvegarde

Les trois jeux préparés sont enregistrés dans `data/`. Le notebook `03_modelisation` les chargera directement. Le dossier `data/` n'est pas envoyé sur GitHub (licence Kaggle).

In [14]:
train.to_csv('../data/train.csv')
val.to_csv('../data/val.csv')
test.to_csv('../data/test.csv')
print('Fichiers enregistrés : data/train.csv, data/val.csv, data/test.csv')

Fichiers enregistrés : data/train.csv, data/val.csv, data/test.csv


## 7. Bilan : chaque problème de l'exploration et son traitement

| # | Problème (exploration) | Traitement | Pourquoi |
|---|---|---|---|
| 1 | Cible déséquilibrée (6,7 %) | Séparation **stratifiée** 60 / 20 / 20 | Même proportion de défauts dans chaque jeu |
| 2 | Revenu manquant (~20 %) | Indicateur `revenu_manquant` + médiane du train | Ces clients font moins défaut : on garde l'information |
| 3 | Personnes à charge manquantes | Médiane du train (0) | Mêmes clients que le revenu manquant : l'indicateur existe déjà |
| 4 | Codes 96 / 98 | Indicateur `retards_code_special` + médiane (0) | Pas de vrais nombres, mais un signal fort (> 50 % de défauts) |
| 5 | Âge = 0 | Médiane du train | Erreur de saisie |
| 6 | Utilisation du crédit > 10 / > 2 | > 10 : médiane ; sinon plafond à 2 | > 10 = erreurs (risque moyen) ; 2 à 10 = trop peu de clients, risque élevé comme au-dessus de 1 |
| 7 | `DebtRatio` en $ sans revenu | Médiane du train quand le revenu n'est pas fiable | Le recalcul ne prédit rien ; les indicateurs suffisent |
| 8 | Revenus de 0 ou 1 ; revenus énormes | Indicateur `revenu_0_1` + médiane ; plafond au 99ᵉ percentile | 0-1 = valeur par défaut ; valeurs énormes peu fiables |
| 9 | Comptages extrêmes | Plafond au 99ᵉ percentile du train | Équivalent de la tranche « N et + » |
| 10 | Lignes en double | Supprimées avant la séparation | Évite qu'une même ligne soit dans le train et le test |

**Ce qui n'est pas fait ici :** le découpage en tranches et le calcul du WoE pour la régression logistique. Ces étapes font partie de la modélisation (notebook 03). Elles seront elles aussi calculées sur le train uniquement.